# CROPSTATE Colab Workflow

Notebook này dùng cho workflow:

- Code clone từ GitHub
- Dataset đọc từ Google Drive: `MyDrive/CROPSTATE_DATASET`
- Knowledge base đọc từ Google Drive: `MyDrive/CROPSTATE_KNOWLEDGE_BASDE`
- Training outputs lưu vào Google Drive: `MyDrive/CROPSTATE_RESULTS`

Trước khi chạy training, vào `Runtime > Change runtime type` và chọn GPU.

In [ ]:
# Sửa REPO_URL thành GitHub repo thật của bạn.
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
PROJECT_DIR = "/content/CROPSTATE_Full_Research_Package"

DATA_ROOT = "/content/drive/MyDrive/CROPSTATE_DATASET"
KNOWLEDGE_ROOT = "/content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASDE"
RESULTS_ROOT = "/content/drive/MyDrive/CROPSTATE_RESULTS"

print("REPO_URL:", REPO_URL)
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("KNOWLEDGE_ROOT:", KNOWLEDGE_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Clone hoặc update repo từ GitHub

In [ ]:
import os
import subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
print("Current directory:", os.getcwd())

## 3. Cài dependencies

In [ ]:
!pip install -r requirements.txt
!pip install -e .

## 4. Kiểm tra dataset và knowledge base trên Drive

In [ ]:
!ls -lah "{DATA_ROOT}"
!ls -lah "{KNOWLEDGE_ROOT}"

Dataset nên có cấu trúc:

```text
CROPSTATE_DATASET/
  01_establishment/
  02_tillering/
  03_stem_booting/
  04_reproductive/
  05_grain_filling/
  06_ripening/
```

Knowledge base nên có `CROPSTATE_Sample_Knowledge_Base.xlsx` hoặc CSV exports trong `CROPSTATE_KNOWLEDGE_BASDE/`.

## 5. Build manifest pilot từ folder dataset

In [ ]:
!PYTHONPATH=src python scripts/build_stage_manifest.py \
  --data-root "{DATA_ROOT}" \
  --output data/stage_folder_manifest.csv

## 6. Convert image manifest từ knowledge base

Nếu `Image_Manifest_Template` vẫn là template placeholder, `data/image_manifest.csv` sẽ rỗng và row bị loại sẽ nằm trong `data/image_manifest_excluded.csv`.

In [ ]:
!PYTHONPATH=src python scripts/convert_image_manifest.py \
  --knowledge-root "{KNOWLEDGE_ROOT}" \
  --data-root "{DATA_ROOT}" \
  --output data/image_manifest.csv

## 7. Convert knowledge chunks

In [ ]:
!PYTHONPATH=src python scripts/convert_knowledge_base.py \
  --knowledge-root "{KNOWLEDGE_ROOT}" \
  --output data/knowledge_chunks.jsonl

## 8. Audit manifest pilot từ folder

In [ ]:
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest data/stage_folder_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --checksum

## 9. Audit manifest từ sheet nếu đã có dữ liệu thật

Chỉ dùng manifest này để train khi `data/image_manifest.csv` có row thật và audit pass.

In [ ]:
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest data/image_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --checksum

## 10. Fine-tune bằng manifest pilot từ folder

Output được lưu thẳng vào Google Drive.

In [ ]:
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest data/stage_folder_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --output "{RESULTS_ROOT}/vision_resnet18_finetune"

## 11. Fine-tune bằng manifest từ sheet

Chỉ chạy cell này nếu `data/image_manifest.csv` đã có row thật.

In [ ]:
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest data/image_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --output "{RESULTS_ROOT}/vision_resnet18_manifest_finetune"

## 12. Fine-tune tiếp từ checkpoint cũ

Dùng khi đã có `best_checkpoint.pt` từ lần train trước.

In [ ]:
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest data/stage_folder_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --resume-checkpoint "{RESULTS_ROOT}/vision_resnet18_finetune/best_checkpoint.pt" \
  --freeze-backbone-epochs 0 \
  --learning-rate 0.0001 \
  --backbone-learning-rate 0.00001 \
  --output "{RESULTS_ROOT}/vision_resnet18_finetune_round2"

## 13. Xem output đã lưu trên Drive

In [ ]:
!ls -lah "{RESULTS_ROOT}/vision_resnet18_finetune"

## 14. Test một ảnh upload từ máy

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))
print("Uploaded:", image_path)

In [ ]:
!PYTHONPATH=src python scripts/predict_image.py \
  --checkpoint "{RESULTS_ROOT}/vision_resnet18_finetune/best_checkpoint.pt" \
  --image "{image_path}"

## Ghi chú

- GitHub chỉ lưu code và metadata nhỏ.
- Google Drive lưu dataset, knowledge base, checkpoint và kết quả training.
- Không commit các file `.pt` lên GitHub.